# V06 — Dash: Interactive Applications

**Dash turns Plotly charts into interactive web applications — no JavaScript required.**
Every Dash app has two parts:
1. **Layout** — what the app looks like (HTML components + Plotly charts)
2. **Callbacks** — what happens when the user interacts (Python functions triggered by inputs)

**Running in Jupyter:** Each exercise runs `app.run(jupyter_mode='inline', port=XXXX)`. Use a different port per exercise to avoid conflicts. **Interrupt the kernel cell to stop an app before running the next one.**

**Reference:** [Dash docs](https://dash.plotly.com/)

**Allowed:** `dash`, `dash.dcc`, `dash.html`, `plotly.graph_objects`, `plotly.express`, `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State, callback_context
import dash_bootstrap_components as dbc
from sklearn.datasets import fetch_openml, fetch_california_housing

# --- Datasets (load once, reuse across all exercises) ---
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail['DayOfWeek'] = retail['InvoiceDate'].dt.day_name()
retail['CustomerID'] = retail['CustomerID'].astype(int)

monthly = retail.groupby('Month').agg(
    Revenue=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique')
).reset_index()
monthly['AvgOrderValue'] = (monthly['Revenue'] / monthly['Orders']).round(2)

housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]

credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')

countries = sorted(retail['Country'].unique())
months = sorted(retail['Month'].unique())
print("Datasets ready.")

---
## Exercise 1 — Hello Dash: Layout Basics

**Spec:** Build the simplest useful Dash app — a static revenue chart with styled layout.

Requirements:
- `html.H1` title: `'Retail Revenue Dashboard'`, centered, color `'#1565C0'`
- `html.P` subtitle: `'Monthly revenue for UK Online Retail 2010–2011'`, grey, font-size 14
- `dcc.Graph` showing the monthly revenue line chart (use `fig` from previous notebooks or rebuild)
- `html.Hr()` divider
- A `html.Div` with a grey background `#F5F5F5`, padding 20px containing a `html.P` with total revenue formatted as `'Total Revenue: £X,XXX,XXX'`
- `app.layout` must be a `html.Div` wrapping everything
- Background color: `white`
- Port: `8051`

In [ ]:
total_rev = retail['Revenue'].sum()

# YOUR CODE HERE
app1 = Dash(__name__)

# Build layout and figure here
app1.layout = None  # replace with your html.Div

app1.run(jupyter_mode='inline', port=8051)

In [ ]:
# --- ASSERTIONS ---
layout = app1.layout
assert layout is not None, "Layout must be defined"
layout_str = str(layout)
assert 'Retail Revenue Dashboard' in layout_str
assert 'dcc.Graph' in layout_str or 'Graph' in layout_str
assert f'£{total_rev:,.0f}' in layout_str or str(int(total_rev)) in layout_str
print("✓ Exercise 1 passed")

---
## Exercise 2 — Dropdown Callback: Dynamic Chart

**Spec:** App with a dropdown that controls which metric to display.

Requirements:
- `dcc.Dropdown` with options: `Revenue`, `Orders`, `Customers`, `AvgOrderValue`
  - `id='metric-dropdown'`, default value `'Revenue'`, `clearable=False`
- `dcc.Graph` with `id='metric-chart'`
- Callback: `Input('metric-dropdown','value')` → updates `Output('metric-chart','figure')`
- Chart: line chart of the selected metric over months
- Title: `f'{selected_metric} Over Time'`
- Chart color changes by metric: Revenue=`'#1565C0'`, Orders=`'#43A047'`, Customers=`'#FB8C00'`, AvgOrderValue=`'#8E24AA'`
- Port: `8052`

In [ ]:
metric_colors = {
    'Revenue': '#1565C0',
    'Orders': '#43A047',
    'Customers': '#FB8C00',
    'AvgOrderValue': '#8E24AA'
}

# YOUR CODE HERE
app2 = Dash(__name__)
app2.layout = None  # replace

# Define callback here using @app2.callback

app2.run(jupyter_mode='inline', port=8052)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app2.layout)
assert 'metric-dropdown' in layout_str
assert 'metric-chart' in layout_str
# Check callback registered
callback_map = app2.callback_map
assert 'metric-chart.figure' in callback_map, "Callback for metric-chart.figure not found"
print("✓ Exercise 2 passed")

---
## Exercise 3 — Multi-Input Callback: Filter + Chart

**Spec:** Filter the scatter chart by country and metric using multiple inputs.

Requirements:
- `dcc.Dropdown` for Country (multi=True, default: top 5 countries)
  - `id='country-filter'`
- `dcc.RadioItems` for chart type: `'Scatter'` or `'Bar'`
  - `id='chart-type'`, default `'Scatter'`
- `dcc.RangeSlider` for revenue range (min=0, max=round up to nearest 1000)
  - `id='revenue-slider'`
- `dcc.Graph` with `id='country-chart'`
- Callback: 3 inputs → 1 output (the figure)
  - Filter `retail` by selected countries and revenue range
  - Build scatter (Revenue vs Orders) or bar (Revenue by Country) based on chart type
- Port: `8053`

In [ ]:
top5 = retail.groupby('Country')['Revenue'].sum().nlargest(5).index.tolist()
country_rev = retail.groupby('Country')['Revenue'].sum().reset_index()
max_rev = int(np.ceil(country_rev['Revenue'].max() / 1000) * 1000)

# YOUR CODE HERE
app3 = Dash(__name__)
app3.layout = None  # replace

# Define callback

app3.run(jupyter_mode='inline', port=8053)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app3.layout)
assert 'country-filter' in layout_str
assert 'chart-type' in layout_str
assert 'revenue-slider' in layout_str
assert 'country-chart' in layout_str
cb = app3.callback_map
assert 'country-chart.figure' in cb
# Callback must have 3 inputs
n_inputs = len(cb['country-chart.figure']['inputs'])
assert n_inputs == 3, f"Expected 3 inputs, got {n_inputs}"
print("✓ Exercise 3 passed")

---
## Exercise 4 — State: Submit Button Pattern

**Spec:** Build an app where the chart only updates when a button is clicked — not on every keystroke.

This is the `State` pattern: inputs are read but don't trigger the callback until the button fires.

Requirements:
- `dcc.Input` for minimum order value (number input, `id='min-order-value'`, default 0)
- `dcc.Input` for maximum order value (number, `id='max-order-value'`, default 10000)
- `html.Button` `'Apply Filter'`, `id='apply-button'`, style it with blue background
- `dcc.Graph` `id='filtered-chart'`
- `html.Div` `id='filter-summary'` showing how many transactions match the filter
- Callback:
  - `Input('apply-button', 'n_clicks')`
  - `State('min-order-value', 'value')`
  - `State('max-order-value', 'value')`
  - Outputs: `filtered-chart.figure` AND `filter-summary.children`
- Chart: histogram of Revenue for transactions in the filtered range
- Port: `8054`

In [ ]:
# YOUR CODE HERE
app4 = Dash(__name__)
app4.layout = None  # replace

# Define callback with State

app4.run(jupyter_mode='inline', port=8054)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app4.layout)
assert 'apply-button' in layout_str
assert 'min-order-value' in layout_str
assert 'max-order-value' in layout_str
assert 'filtered-chart' in layout_str
assert 'filter-summary' in layout_str
cb = app4.callback_map
# Must output to both filtered-chart and filter-summary
output_keys = list(cb.keys())
has_chart = any('filtered-chart' in k for k in output_keys)
has_summary = any('filter-summary' in k for k in output_keys)
assert has_chart and has_summary, "Must output to both chart and summary div"
# Must use State (not Input) for the text inputs
for key in output_keys:
    if 'filtered-chart' in key:
        states = cb[key].get('state', [])
        state_ids = [s.get('id','') for s in states]
        assert 'min-order-value' in state_ids, "min-order-value must be State, not Input"
print("✓ Exercise 4 passed")

---
## Exercise 5 — dcc.Store: Client-Side State

**Spec:** Use `dcc.Store` to cache filtered data so multiple charts can share it without re-filtering.

This is a critical pattern for performance in production Dash apps.

Requirements:
- `dcc.Store(id='filtered-data-store', storage_type='memory')`
- `dcc.Dropdown` for Country selection (multi=True)
- `dcc.DatePickerRange` for date range (`id='date-range'`)
- Two `dcc.Graph` components: `id='chart-revenue'` and `id='chart-orders'`
- Callback 1: Country + Date → Store (serializes filtered DataFrame to JSON)
- Callback 2: Store → Revenue chart
- Callback 3: Store → Orders chart
- Both charts update from the same cached store, not by re-filtering independently
- Port: `8055`

In [ ]:
date_min = retail['InvoiceDate'].min().date()
date_max = retail['InvoiceDate'].max().date()

# YOUR CODE HERE
app5 = Dash(__name__)
app5.layout = None  # replace

# Define 3 callbacks

app5.run(jupyter_mode='inline', port=8055)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app5.layout)
assert 'filtered-data-store' in layout_str
assert 'chart-revenue' in layout_str
assert 'chart-orders' in layout_str
assert 'date-range' in layout_str
cb = app5.callback_map
# Store must be an output of one callback
assert any('filtered-data-store' in k for k in cb.keys()), \
    "filtered-data-store must be an output of a callback"
# Revenue and orders charts must exist as outputs
assert any('chart-revenue' in k for k in cb.keys())
assert any('chart-orders' in k for k in cb.keys())
print("✓ Exercise 5 passed")

---
## Exercise 6 — Interactive Data Table + Chart

**Spec:** Build a dashboard where clicking a row in a data table updates the chart below it.

Requirements:
- Import and use `dash.dash_table.DataTable`
- Table shows top 15 countries: Country, Revenue, Orders, AvgOrderValue
  - `id='country-table'`, `row_selectable='single'`, `selected_rows=[0]`
  - Style: alternate row colors, header bold
- `dcc.Graph` `id='country-detail-chart'`
- Callback: `Input('country-table','selected_rows')` → monthly revenue line chart for the selected country
- Title of chart: `f'Monthly Revenue: {selected_country}'`
- Port: `8056`

In [ ]:
from dash import dash_table

country_summary = (
    retail.groupby('Country')
    .agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'))
    .reset_index()
    .nlargest(15,'Revenue')
)
country_summary['AvgOrderValue'] = (country_summary['Revenue'] / country_summary['Orders']).round(2)
country_summary['Revenue'] = country_summary['Revenue'].round(2)

# YOUR CODE HERE
app6 = Dash(__name__)
app6.layout = None  # replace

# Define callback

app6.run(jupyter_mode='inline', port=8056)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app6.layout)
assert 'country-table' in layout_str
assert 'country-detail-chart' in layout_str
cb = app6.callback_map
assert any('country-detail-chart' in k for k in cb.keys())
detail_key = [k for k in cb.keys() if 'country-detail-chart' in k][0]
inputs = cb[detail_key]['inputs']
input_ids = [i.get('id','') for i in inputs]
assert 'country-table' in input_ids, "Callback must be triggered by country-table"
print("✓ Exercise 6 passed")

---
## Exercise 7 — Tabs: Multi-Page Layout

**Spec:** Build a multi-tab app — each tab shows a different analytical view.

Requirements:
- `dcc.Tabs` with 3 tabs:
  - `'tab-overview'`: label `'📊 Overview'` — KPI tiles + revenue trend
  - `'tab-products'`: label `'📦 Products'` — top products bar chart with dropdown filter for country
  - `'tab-customers'`: label `'👥 Customers'` — customer revenue distribution histogram + segment donut
- `dcc.Graph` components inside each tab's content div
- Callback: `Input('main-tabs','value')` → renders appropriate content for `Output('tab-content','children')`
- Active tab styling: bottom border `'3px solid #1565C0'`
- Port: `8057`

In [ ]:
customer_rev = retail.groupby('CustomerID')['Revenue'].sum()
seg = pd.qcut(customer_rev, q=4, labels=['Low','Mid-Low','Mid-High','High'])
seg_counts = seg.value_counts().sort_index()

# YOUR CODE HERE
app7 = Dash(__name__)
app7.layout = None  # replace

# Define callback

app7.run(jupyter_mode='inline', port=8057)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app7.layout)
assert 'main-tabs' in layout_str
assert 'tab-content' in layout_str
assert 'tab-overview' in layout_str
assert 'tab-products' in layout_str
assert 'tab-customers' in layout_str
cb = app7.callback_map
assert any('tab-content' in k for k in cb.keys()), "Tab content callback missing"
print("✓ Exercise 7 passed")

---
## Exercise 8 — Capstone: Full Analytics Dashboard

**Spec:** Build a production-grade analytics dashboard combining everything from this notebook.

**App: Retail Intelligence Dashboard**

Layout:
- Top bar: `html.H1` title + `dcc.Dropdown` for Country filter (multi, default all) + `dcc.DatePickerRange`
- Row 1: 3 KPI indicator cards (Revenue, Orders, Customers) — update based on filters
- Row 2: Left (60%) Revenue trend + Right (40%) Top Products bar
- Row 3: Left (50%) DOW revenue heatmap + Right (50%) Customer AOV histogram
- Footer: `html.P` with last updated timestamp

Callbacks:
1. Filter Store: Country + DateRange → `dcc.Store`
2. KPIs: Store → 3 KPI values (use `Output` list)
3. Trend chart: Store → line chart
4. Products chart: Store → bar chart
5. DOW heatmap: Store → heatmap
6. AOV histogram: Store → histogram

Styling: clean white cards with box-shadow, `#1565C0` header

- Port: `8058`

In [ ]:
from datetime import datetime

retail['Date'] = retail['InvoiceDate'].dt.date
retail['Hour'] = retail['InvoiceDate'].dt.hour

# YOUR CODE HERE
app8 = Dash(__name__)
app8.layout = None  # replace

# Define all 6 callbacks

app8.run(jupyter_mode='inline', port=8058)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app8.layout)
# Core components present
for component_id in ['filtered-data-store', 'date-range']:
    assert component_id in layout_str or 'Store' in layout_str, \
        f"Component {component_id} missing from layout"

cb = app8.callback_map
# Must have at least 5 callbacks registered
assert len(cb) >= 5, f"Expected >= 5 callbacks, got {len(cb)}"

# At least one Store output (for caching)
store_outputs = [k for k in cb.keys() if 'store' in k.lower() or 'Store' in k]
assert len(store_outputs) >= 1, "Must use dcc.Store for data caching"

print(f"✓ Exercise 8 passed — {len(cb)} callbacks registered")
print("Dashboard is running. Visit the embedded view above.")